# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata attributes
# Note: The metadata is a Python object, not a dictionary
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Dataset Description: {dataset.metadata.description}")
print(f"Dataset Version: {dataset.metadata.version}")
print(f"Date Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, their `@id`s, and list their contained fields (columns).

In [ ]:
# Explore available record sets
print("Record Sets found in the dataset:")
for record_set in dataset.record_sets:
    print(f"- RecordSet Name: {record_set.name}")
    print(f"  @id: {record_set.identifier}")
    print("  Fields (by @id):")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.identifier}, dataType: {field.data_type})")
    print()

### Example: Preview records from the first record set

We can preview records from a record set by referencing its `@id`. (See above for the available record set IDs.)

In [ ]:
# Pick the first record set for demonstration purposes
record_set_id = dataset.record_sets[0].identifier
print(f"Previewing records from RecordSet @id={record_set_id}")
for i, record in enumerate(dataset.records(record_set=record_set_id)):
    print(record)
    if i>=2:  # Only show first 3 records
        break

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Each DataFrame's columns correspond to the field `@id`s defined in the Croissant schema.

In [ ]:
# Gather all record set IDs
record_set_ids = [rset.identifier for rset in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        # Load all records for each record set
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set '{record_set_id}'. Fields: {df.columns.tolist()}")
        else:
            print(f"Record set '{record_set_id}' contains no records.")
    except Exception as e:
        print(f"Could not load record set '{record_set_id}': {e}")

### Preview columns and records
Below, we show the column names and a few sample records for the main record set (set as the first one in the schema).

In [ ]:
main_record_set_id = dataset.record_sets[0].identifier
print(f"Columns in main record set (@id={main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We can now process the loaded data. Typical tasks might include removing outliers, normalizing numeric fields, and grouping by categorical attributes.

Below, choose a record set, a numeric field (by its `@id`), and a grouping field (by its `@id`) for analysis. For this dataset, likely fields for numeric analysis are protein abundance, molecular weight, or peptide count.

In [ ]:
# Identify a numeric field and a group field
df = dataframes[main_record_set_id]

# Use schema to find numeric fields (example: find 'molecular_weight', 'peptide_count', or 'abundance')
sample_field_ids = [field.identifier for field in dataset.record_sets[0].fields if field.data_type in ['Float', 'Number', 'Integer']]
print("Available numeric fields (@id):", sample_field_ids)

# Choose one numeric field for filtering/normalization
numeric_field_id = sample_field_ids[0] if sample_field_ids else None
print("Using numeric field for demo:", numeric_field_id)

# Choose a group field, try to find a 'description' or categorical type field
group_field_id = None
for field in dataset.record_sets[0].fields:
    if field.data_type in ['Text', 'String'] and 'desc' in field.identifier.lower():
        group_field_id = field.identifier
        break
group_field_id = group_field_id or sample_field_ids[0]  # fallback
print("Group field:", group_field_id)

# Proceed if numeric field exists
if numeric_field_id is not None:
    # Remove nulls and outliers
    threshold = df[numeric_field_id].quantile(0.99)  # use 99th for threshold
    filtered_df = df[df[numeric_field_id] < threshold]

    print(f"Filtered records where {numeric_field_id} < {threshold}: count =", len(filtered_df))
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group field if it exists
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field in the main record set, and (if possible) show grouped means by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Plot grouped means if grouped data available
    if group_field_id and group_field_id in df.columns:
        means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10,4))
        sns.barplot(x=means.values, y=means.index)
        plt.title(f"Top 10 Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion

- This notebook used `mlcroissant` to load the FAIR^2 dataset and explored the available record sets and their schema. All entities (record sets, fields, columns) were referenced by their `@id` as defined in the dataset's Croissant JSON-LD schema.
- Numeric data columns such as protein abundance or molecular weight can be analyzed and visualized directly after loading via schema-safe methods.
- For further study, we recommend referencing the dataset documentation and schema for precise field meanings and full record set hierarchy.